# Анализ лояльности пользователей Яндекс Афиши


## Цели и задачи проекта:

С помощью датасета с информацией о клиентах и их активности, надо проанализировать и выявить портрет пользователя, который с большей вероятностью будет возвращаться на платформу и делать повторные заказы. Это поможет оптимизировать рекламные бюжеты и влиять на продолжительность удержания клиента

## Описание данных:
- user_id — уникальный идентификатор пользователя, совершившего заказ;
- device_type_canonical — тип устройства, с которого был оформлен заказ (mobile — мобильные устройства, desktop — стационарные);
- order_id — уникальный идентификатор заказа;
- order_dt — дата создания заказа (используйте данные created_dt_msk);
- order_ts — дата и время создания заказа (используйте данные created_ts_msk);
- currency_code — валюта оплаты;
- revenue — выручка от заказа;
- tickets_count — количество купленных билетов;
- days_since_prev — количество дней от предыдущей покупки пользователя, для пользователей с одной покупкой — значение пропущено;
- event_id — уникальный идентификатор мероприятия;
- service_name — название билетного оператора;
event_type_main — основной тип мероприятия (театральная постановка, концерт и так далее);
- region_name — название региона, в котором прошло мероприятие;
- city_name — название города, в котором прошло мероприятие.


### 0.Установка необходимых библиотек

In [1]:
#Установка необходимых библиотек
!pip install sqlalchemy

In [2]:
!pip install psycopg2

In [3]:
#Импорт необходимых библиотек для работы
#Для анализа данных
import pandas as pd

#Для визуализации
import matplotlib.pyplot as plt
import seaborn as sns

#Для подключения к БД
import os 
from sqlalchemy import create_engine
from dotenv import load_dotenv

In [4]:
#Получаю доступ к БД, используя данные из файла .env
db_config = {
    'user': os.getenv('DB_USER'),
    'pwd': os.getenv('DB_PASSWORD'),
    'host': os.getenv('DB_HOST'),
    'port': os.getenv('DB_PORT'),
    'db': os.getenv('DB_NAME')
}

In [5]:
#Создаю данные для подключения
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

In [6]:
#Проверяю результат 
db_config

{'user': 'praktikum_student',
 'pwd': 'Sdf4$2;d-d30pp',
 'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
 'port': '6432',
 'db': 'data-analyst-afisha'}

In [7]:
engine = create_engine(connection_string)

In [10]:
#Импортируем SQL-запрос 
query = '''
-- Настройка параметра synchronize_seqscans важна для проверки
WITH set_config_precode AS (
  SELECT set_config('synchronize_seqscans', 'off', true)
)
-- Напишите ваш запрос ниже
SELECT p.user_id,
      p.device_type_canonical,
      p.order_id,
      p.created_dt_msk AS order_dt,
      p.created_ts_msk AS order_ts,
      p.currency_code,
      p.revenue,
      p.tickets_count,
      EXTRACT (DAY FROM (p.created_dt_msk - LAG(p.created_dt_msk) OVER (PARTITION BY p.user_id ORDER BY p.created_dt_msk)))::INTEGER AS days_since_prev,
      p.event_id,
      e.event_name_code AS event_name,
      e.event_type_main,
      p.service_name,
      r.region_name,
      c.city_name
FROM afisha.purchases p
JOIN afisha.events e USING(event_id)
JOIN afisha.city c USING(city_id)
JOIN afisha.regions r USING(region_id)
WHERE p.device_type_canonical IN ('mobile', 'desktop') AND e.event_type_main NOT IN ('фильм')
ORDER BY p.user_id;
''' 

In [11]:
#Присваиваю результат SQL-запроса в переменную
df = pd.read_sql_query(query, con=engine)


In [12]:
#Проверяю результат выгрузки датасета
df.head()

,user_id,device_type_canonical,order_id,order_dt,order_ts,currency_code,revenue,tickets_count,days_since_prev,event_id,event_name,event_type_main,service_name,region_name,city_name
0,0002849b70a3ce2,mobile,4359165,2024-08-20,2024-08-20 16:08:03,rub,1521.94,4,NaN,169230,f0f7b271-04eb-4af6-bcb8-8f05cf46d6ad,театр,Край билетов,Каменевский регион,Глиногорск
1,0005ca5e93f2cf4,mobile,7965605,2024-07-23,2024-07-23 18:36:24,rub,289.45,2,NaN,237325,40efeb04-81b7-4135-b41f-708ff00cc64c,выставки,Мой билет,Каменевский регион,Глиногорск
2,0005ca5e93f2cf4,mobile,7292370,2024-10-06,2024-10-06 13:56:02,rub,1258.57,4,75.0,578454,01f3fb7b-ed07-4f94-b1d3-9a2e1ee5a8ca,другое,За билетом!,Каменевский регион,Глиногорск
3,000898990054619,mobile,1139875,2024-07-13,2024-07-13 19:40:48,rub,8.49,2,NaN,387271,2f638715-8844-466c-b43f-378a627c419f,другое,Лови билет!,Североярская область,Озёрск
4,000898990054619,mobile,972400,2024-10-04,2024-10-04 22:33:15,rub,1390.41,3,83.0,509453,10d805d3-9809-4d8a-834e-225b7d03f95d,стендап,Билеты без проблем,Озернинский край,Родниковецк


### 1. Знакомство с данными